# Diabetes Dataset Exploration

In [1]:
from __future__ import annotations

import polars as pl
from config import get_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from sklearn import datasets

from dfkit import DataFrameToolkit, enable_logging
from dfkit.tool_modules import DecisionTreeModule

/Users/nlibertini/code/dfkit/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## Load dataset as a dataframe

In [2]:
data, target = datasets.load_diabetes(return_X_y=True, scaled=False)
df = pl.DataFrame(
    data=data,
    schema=["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"],
)

df = df.with_columns(
    pl.col("sex").map_elements(lambda x: "male" if x == 1 else "female", return_dtype=pl.String),
    pl.Series(target).alias("disease_progression"),
)

df

age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,disease_progression
f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
59.0,"""female""",32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0,151.0
48.0,"""male""",21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0,75.0
72.0,"""female""",30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0,141.0
24.0,"""male""",25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0,206.0
50.0,"""male""",23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0,135.0
…,…,…,…,…,…,…,…,…,…,…
60.0,"""female""",28.2,112.0,185.0,113.8,42.0,4.0,4.9836,93.0,178.0
47.0,"""female""",24.9,75.0,225.0,166.0,42.0,5.0,4.4427,102.0,104.0
60.0,"""female""",24.9,99.67,162.0,106.6,43.0,3.77,4.1271,95.0,132.0


## Register the dataset with the toolkit

In [3]:
# Initialize the toolkit
toolkit = DataFrameToolkit()

# Register the diabetes dataset with the toolkit
_ = toolkit.register_dataframe(
    name="Diabetes Progression Dataset",
    dataframe=df,
    description="""
    Ten baseline variables, age, sex, body mass index, average blood pressure,
    and six blood serum measurements were obtained for each diabetes patient,
    as well as the response of interest, a quantitative measure of disease
    progression one year after baseline.
    """,
    column_descriptions={
        "age": "Age of the patient in years.",
        "sex": "Sex of the patient",
        "bmi": "Body mass index.",
        "bp": "Average blood pressure.",
        "s1": "TC, total serum cholesterol.",
        "s2": "LDL, low-density lipoproteins.",
        "s3": "HDL, high-density lipoproteins.",
        "s4": "TCH, total cholesterol / HDL.",
        "s5": "LTG, possibly log of serum triglycerides level.",
        "s6": "GLU, blood sugar level.",
        "disease_progression": "A quantitative measure of disease progression one year after baseline.",
    },
)

# View the registered dataset as a markdown table
print(toolkit.view_as_markdown_table("Diabetes Progression Dataset"))

| age  | sex    | bmi  | bp    | s1    | s2    | s3   | s4   | s5     | s6    | disease_progressio |
|      |        |      |       |       |       |      |      |        |       | n                  |
|------|--------|------|-------|-------|-------|------|------|--------|-------|--------------------|
| 59.0 | female | 32.1 | 101.0 | 157.0 | 93.2  | 38.0 | 4.0  | 4.8598 | 87.0  | 151.0              |
| 48.0 | male   | 21.6 | 87.0  | 183.0 | 103.2 | 70.0 | 3.0  | 3.8918 | 69.0  | 75.0               |
| 72.0 | female | 30.5 | 93.0  | 156.0 | 93.6  | 41.0 | 4.0  | 4.6728 | 85.0  | 141.0              |
| 24.0 | male   | 25.3 | 84.0  | 198.0 | 131.4 | 40.0 | 5.0  | 4.8903 | 89.0  | 206.0              |
| 50.0 | male   | 23.0 | 101.0 | 192.0 | 125.4 | 52.0 | 4.0  | 4.2905 | 80.0  | 135.0              |
| …    | …      | …    | …     | …     | …     | …    | …    | …      | …     | …                  |
| 60.0 | female | 28.2 | 112.0 | 185.0 | 113.8 | 42.0 | 4.0  | 4.9836 | 93.0  | 178.0      

## Create an agent with the toolkit

In [4]:
df_agent = create_agent(
    model=get_chat_model(),
    # Core tools + Decision Tree tools
    tools=toolkit.get_tools(DecisionTreeModule),
    # Prompt for core tools + Decision Tree tools
    system_prompt=toolkit.get_system_prompt(DecisionTreeModule),
)

# Print the combined system prompt
print(toolkit.get_system_prompt(DecisionTreeModule))

You have access to a DataFrame toolkit with the following core tools:

- **list_dataframes**: List all available DataFrames with their names, IDs, and schema information.
- **get_dataframe_id**: Get the unique ID for a DataFrame by its human-readable name. Use IDs in SQL queries.
- **get_dataframe_reference**: Get detailed schema information about a DataFrame by name or ID.
- **execute_sql**: Execute a SQL SELECT query against registered DataFrames. Use DataFrame IDs as table names.
- **view_as_markdown_table**: View a DataFrame as a formatted markdown table for data inspection.

Workflow: First use list_dataframes to discover available data, then get_dataframe_id to get IDs, then execute_sql to query using those IDs.

## DecisionTreeModule
You have access to an **analyze_with_decision_tree** tool that discovers relationships between columns in a DataFrame by fitting a decision tree.

**When to use it**: Use this tool for insight discovery - to understand which features best predict or

## Ask questions about the dataset

### Q1

In [5]:
num_dataframes = len(toolkit.list_dataframes())
with enable_logging():
    response = df_agent.invoke({
        "messages": [
            HumanMessage("What's the relationship between BMI and disease progression in the diabetes dataset?")
        ]
    })
    messages = response.get("messages", [])
    if messages:
        last_message = messages[-1]
        print(last_message.content)
        print(f"# of dataframes created: {len(toolkit.list_dataframes()) - num_dataframes}")
        num_dataframes = len(toolkit.list_dataframes())

2026-03-27 09:46:34.521 | TOOL_CALL | list_dataframes - Tool call: list_dataframes
2026-03-27 09:46:36.550 | TOOL_CALL | analyze_with_decision_tree - Tool call: analyze_with_decision_tree for dataframe=Diabetes Progression Dataset, target=disease_progression
2026-03-27 09:46:38.046 | TOOL_CALL | get_dataframe_id - Tool call: get_dataframe_id
2026-03-27 09:46:39.990 | TOOL_CALL | execute_sql - Tool call: execute_sql
2026-03-27 09:46:39.995 | WARNING  | execute_sql - Tool call error: execute_sql
2026-03-27 09:46:41.425 | TOOL_CALL | execute_sql - Tool call: execute_sql
2026-03-27 09:46:41.428 | WARNING  | execute_sql - Tool call error: execute_sql
2026-03-27 09:46:42.459 | TOOL_CALL | execute_sql - Tool call: execute_sql
2026-03-27 09:46:43.366 | TOOL_CALL | view_as_markdown_table - Tool call: view_as_markdown_table
2026-03-27 09:46:48.617 | TOOL_CALL | list_dataframes - Tool call: list_dataframes
2026-03-27 09:46:48.618 | TOOL_CALL | list_dataframes - Tool call: list_dataframes


## Summary: BMI and Disease Progression Relationship

Based on the analysis of the Diabetes Progression Dataset (442 patients), here are the key findings:

### **Strong Positive Relationship**
There is a **clear positive correlation between BMI and disease progression**:

- **Model Performance**: The decision tree explains **43% of the variance** (R² = 0.43) in disease progression using only BMI, with an average prediction error of ±58 points.

- **Progressive Pattern**: Disease progression increases substantially as BMI increases:
  - **Low BMI (≤19.35)**: Average disease progression ≈ **99**
  - **Moderate BMI (20-25)**: Average disease progression ≈ **112-174**
  - **High BMI (28-30)**: Average disease progression ≈ **196-229**
  - **Very High BMI (>35.8)**: Average disease progression ≈ **286**

### **Key Insights**

1. **Non-linear Relationship**: The relationship isn't perfectly linear—there's some variability, but the overall trend is clear: higher BMI is associated with greater

### Q2

In [ ]:
with enable_logging():
    response = df_agent.invoke({
        "messages": [
            HumanMessage("Is there a significant difference in disease progression between male and female patients?")
        ]
    })
    messages = response.get("messages", [])
    if messages:
        last_message = messages[-1]
        print(last_message.content)
        print(f"# of dataframes created: {len(toolkit.list_dataframes()) - num_dataframes}")
        num_dataframes = len(toolkit.list_dataframes())

### Q3

In [ ]:
with enable_logging():
    response = df_agent.invoke({
        "messages": [
            HumanMessage(
                "Ignoring age, sex, and race what are the biggest factors impacting in disease progression?"
                " How would you characterize a low risk person, a medium risk person, and a high risk person?"
            )
        ]
    })
    messages = response.get("messages", [])
    if messages:
        last_message = messages[-1]
        print(last_message.content)
        print(f"# of dataframes created: {len(toolkit.list_dataframes()) - num_dataframes}")
        num_dataframes = len(toolkit.list_dataframes())

### Q4

In [ ]:
with enable_logging():
    response = df_agent.invoke({
        "messages": [
            HumanMessage(
                "How does race impact disease progression?"
            )
        ]
    })
    messages = response.get("messages", [])
    if messages:
        last_message = messages[-1]
        print(last_message.content)
        print(f"# of dataframes created: {len(toolkit.list_dataframes()) - num_dataframes}")
        num_dataframes = len(toolkit.list_dataframes())

### Q5

In [ ]:
 with enable_logging():
    response = df_agent.invoke({
        "messages": [
            HumanMessage(
                "Explore this dataset. What are some critical findings?"
                " What are some subtle, but interesting relationship?"
                " What would be some good data to gather to enrich our analysis?"
            )
        ]
    })
    messages = response.get("messages", [])
    if messages:
        last_message = messages[-1]
        print(last_message.content)
        print(f"# of dataframes created: {len(toolkit.list_dataframes()) - num_dataframes}")
        num_dataframes = len(toolkit.list_dataframes())